<a href="https://colab.research.google.com/github/RCalvoso/grupo4_projeto_integrador_2/blob/main/05_PLN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import os
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel
from google.colab import drive
from tqdm import tqdm

# 1. Garantir acesso aos arquivos
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# Localizar diretório dos dados
dir_path = None
for root, dirs, files in os.walk('/content/drive/MyDrive'):
    if 'train_clean.csv' in files:
        dir_path = root
        break

train_df = pd.read_csv(os.path.join(dir_path, 'train_clean.csv'))
val_df = pd.read_csv(os.path.join(dir_path, 'val_clean.csv'))
test_df = pd.read_csv(os.path.join(dir_path, 'test_clean.csv'))

# 2. Carregar o Modelo BERTimbau (BERT em Português)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Utilizando dispositivo: {device}")

model_name = "neuralmind/bert-base-portuguese-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name).to(device)
model.eval()

# 3. Função para extrair Embeddings de Texto via BERTimbau
def extract_text_embeddings(df, batch_size=32):
    # Concatena título e descrição
    texts = (df['title'].fillna('') + " " + df['description'].fillna('')).tolist()
    all_embeddings = []

    with torch.no_grad():
        for i in tqdm(range(0, len(texts), batch_size), desc="Gerando Embeddings"):
            batch_texts = texts[i:i + batch_size]
            encoded = tokenizer(
                batch_texts,
                padding=True,
                truncation=True,
                max_length=128,
                return_tensors='pt'
            ).to(device)

            outputs = model(**encoded)
            # Utiliza o Mean Pooling da última camada oculta como embedding
            mask = encoded['attention_mask'].unsqueeze(-1)
            token_embeddings = outputs.last_hidden_state
            sum_embeddings = torch.sum(token_embeddings * mask, dim=1)
            sum_mask = torch.clamp(mask.sum(dim=1), min=1e-9)
            mean_pooled = (sum_embeddings / sum_mask).cpu().numpy()

            all_embeddings.append(mean_pooled)

    return np.vstack(all_embeddings)

print("\nExtraindo embeddings de texto de Treino...")
text_embeds_train = extract_text_embeddings(train_df)

print("\nExtraindo embeddings de texto de Validação...")
text_embeds_val = extract_text_embeddings(val_df)

print("\nExtraindo embeddings de texto de Teste...")
text_embeds_test = extract_text_embeddings(test_df)

# 4. Salvar os embeddings de texto gerados
np.save(os.path.join(dir_path, 'text_embeds_train.npy'), text_embeds_train)
np.save(os.path.join(dir_path, 'text_embeds_val.npy'), text_embeds_val)
np.save(os.path.join(dir_path, 'text_embeds_test.npy'), text_embeds_test)

print(f"\n✅ Embeddings textuais (PLN - BERTimbau) gerados com sucesso!")
print(f"Dimensão das feições de texto: {text_embeds_train.shape}")

Utilizando dispositivo: cpu


config.json:   0%|          | 0.00/647 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/43.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/210k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

...


Extraindo embeddings de texto de Validação...


Gerando Embeddings: 100%|██████████| 24/24 [01:19<00:00,  3.31s/it]



Extraindo embeddings de texto de Teste...


Gerando Embeddings: 100%|██████████| 24/24 [01:21<00:00,  3.41s/it]


✅ Embeddings textuais (PLN - BERTimbau) gerados com sucesso!
Dimensão das feições de texto: (3500, 768)
